# 4 — Visualization & Diagnostics (Fixed)

**Perbaikan dari audit:**
- ✅ Multi-province visualization (bukan cherry-picked 1 provinsi)
- ✅ Residual diagnostic plot
- ✅ Correlation heatmap
- ✅ Distribution plots
- ❌ Tidak ada causal claim ("Domino Effect") tanpa bukti statistik

**Input:**
- `3_modelling/output/3_model_predictions.csv`
- `3_modelling/output/per_province_metrics.csv`
- `2_data_preprocessing/output/2.2_final_feature_set.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '3_modelling' / 'output' / '3_model_predictions.csv').exists():
            return p
    raise FileNotFoundError('Could not find 3_model_predictions.csv')

ROOT = find_project_root(Path.cwd())
output_dir = ROOT / '4_visualization' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

df_pred = pd.read_csv(ROOT / '3_modelling' / 'output' / '3_model_predictions.csv')
df_pred['tanggal'] = pd.to_datetime(df_pred['tanggal'])

df_feat = pd.read_csv(ROOT / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv')
df_feat['tanggal'] = pd.to_datetime(df_feat['tanggal'])

prov_metrics_path = ROOT / '3_modelling' / 'output' / 'per_province_metrics.csv'
if prov_metrics_path.exists():
    df_prov_m = pd.read_csv(prov_metrics_path)

metrics_path = ROOT / '3_modelling' / 'output' / 'model_metrics.csv'
if metrics_path.exists():
    print('=== Model Metrics ===')
    display(pd.read_csv(metrics_path))

print(f'Predictions: {len(df_pred):,} rows | Features: {len(df_feat):,} rows')

In [ ]:
# === 1. Pred vs Actual — SEMUA provinsi (sample 6) ===
sample_provs = df_pred['provinsi_id'].unique()[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
axes = axes.flatten()

for i, prov_id in enumerate(sample_provs):
    pdata = df_pred[df_pred['provinsi_id'] == prov_id].sort_values('tanggal')
    ax = axes[i]
    ax.plot(pdata['tanggal'], pdata['y_true'], label='Actual', lw=2)
    ax.plot(pdata['tanggal'], pdata['y_pred'], label='Predicted', lw=2, ls='--')
    ax.set_title(f'{pdata["nama_provinsi"].iloc[0]} (ID={prov_id})', fontsize=9)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle('Prediction vs Actual (Test Set 2025) — 6 Provinces', fontweight='bold')
fig.tight_layout()
fig.savefig(output_dir / 'pred_vs_actual_multi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 2. Residual Diagnostics ===
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residual distribution
axes[0].hist(df_pred['error'], bins=40, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', ls='--')
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Error (pred - actual)')

# Residual vs predicted
axes[1].scatter(df_pred['y_pred'], df_pred['error'], alpha=0.4, s=15)
axes[1].axhline(0, color='red', ls='--')
axes[1].set_title('Residual vs Predicted')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Error')

# Actual vs Predicted scatter
axes[2].scatter(df_pred['y_true'], df_pred['y_pred'], alpha=0.4, s=15)
lims = [min(df_pred['y_true'].min(), df_pred['y_pred'].min()),
        max(df_pred['y_true'].max(), df_pred['y_pred'].max())]
axes[2].plot(lims, lims, 'r--', label='Perfect prediction')
axes[2].set_title('Actual vs Predicted')
axes[2].set_xlabel('Actual')
axes[2].set_ylabel('Predicted')
axes[2].legend(fontsize=8)

fig.tight_layout()
fig.savefig(output_dir / 'residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 3. Per-Province RMSE bar chart ===
if 'df_prov_m' in dir():
    fig, ax = plt.subplots(figsize=(14, 6))
    prov_sorted = df_prov_m.sort_values('rmse_model', ascending=True)
    y_pos = range(len(prov_sorted))
    ax.barh(y_pos, prov_sorted['rmse_model'], label='Model RMSE', alpha=0.8)
    ax.barh(y_pos, prov_sorted['rmse_naive'], label='Naive RMSE', alpha=0.4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(prov_sorted['nama_provinsi'], fontsize=7)
    ax.set_xlabel('RMSE')
    ax.set_title('Per-Province RMSE: Model vs Naive Baseline')
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / 'per_province_rmse.png', dpi=150, bbox_inches='tight')
    plt.show()

# === 4. TWP90 Distribution ===
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_feat['twp90_pct'], bins=50, edgecolor='black', alpha=0.7)
ax.set_title('Distribution of TWP90 (%)')
ax.set_xlabel('twp90_pct')
ax.axvline(df_feat['twp90_pct'].median(), color='red', ls='--', label=f'Median: {df_feat["twp90_pct"].median():.4f}')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'twp90_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 5. Correlation Heatmap (fitur utama) ===
core_cols = ['twp90_pct', 'x1_bi_rate_pct', 'x2_inflasi_yoy',
             'x3_pdrb_per_kapita', 'x4_tpt_pct',
             'x5_penetrasi_internet_pct', 'x8_ldr_pct',
             'x9_npl_ratio', 'x10_rasio_umkm']
core_cols = [c for c in core_cols if c in df_feat.columns]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_feat[core_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Correlation Heatmap (Core Features)')
fig.tight_layout()
fig.savefig(output_dir / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAll visualizations saved to:', output_dir)